In [15]:
import pandas as pd
import numpy as np  
import altair as alt
import urllib.request
import json
import os

In [16]:
# Read the killed and injured in road accidents data
ki_road_accidents = pd.read_csv('../data/Killed and injured in road accidents (IT1,41_270_DF_DCIS_MORTIFERITISTR1_1,1.0).csv')
ki_road_accidents.head()

,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,ACCIDENT_LOCALIZATON,Localization of the accident,INTERSECTION,Intersection (DESC),...,PERSON_CLASS,Person class,AGE,Age (DESC),SEX,Sex (DESC),MONTH,Month (DESC),TIME_PERIOD,Observation
0,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2010,233
1,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2011,206
2,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2012,219
3,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2013,184
4,A,Annual,ITC1,Piemonte,KILLINJ,Killed and injured,9,Total,9,Total,...,C,Driver,TOTAL,Total,9,Total,99,Total,2014,179


In [17]:
# Read both Italy regions provinces CSV files
pop_df1 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0).csv')
pop_df2 = pd.read_csv('../data/Italy, regions, provinces (IT1,22_289_DF_DCIS_POPRES1_1,1.0) (1).csv')

# Combine them into one population dataframe
population_df = pd.concat([pop_df1, pop_df2], ignore_index=True)

# Sort by territory and time period for better organization
population_df = population_df.sort_values(['REF_AREA', 'TIME_PERIOD']).reset_index(drop=True)

print(f"Combined population dataframe shape: {population_df.shape}")
population_df.head()

Combined population dataframe shape: (154, 16)


,FREQ,Frequency,REF_AREA,Territory,DATA_TYPE,Indicator,SEX,Gender,AGE,Age (DESC),MARITAL_STATUS,Marital status,TIME_PERIOD,Observation,OBS_STATUS,Observation status
0,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2019,4328565,NaN,NaN
1,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2020,4311217,NaN,NaN
2,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2021,4274945,NaN,NaN
3,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2022,4256350,NaN,NaN
4,A,Annual,ITC1,Piemonte,JAN,Population on 1st January,9,Total,TOTAL,Total,99,Total,2023,4251351,NaN,NaN


In [18]:
# Prepare data for the chart - filter for relevant data
# Exclude 'Total' values to get individual categories
chart_data = ki_road_accidents[
    (ki_road_accidents['Result (DESC)'].isin(['Killed', 'Injured'])) &
    (ki_road_accidents['Person class'].isin(['Driver', 'Passenger', 'Pedestrian'])) &
    (ki_road_accidents['Age (DESC)'] == 'Total') &
    (ki_road_accidents['Sex (DESC)'] == 'Total') &
    (ki_road_accidents['Month (DESC)'] == 'Total') &
    (ki_road_accidents['Intersection (DESC)'] == 'Total') &
    (ki_road_accidents['Localization of the accident'] == 'Total') &
    (ki_road_accidents['Road accident type'] == 'Total')
].copy()

# Group by year, result type, person class, and region
grouped_data = chart_data.groupby([
    'TIME_PERIOD', 
    'Result (DESC)', 
    'Person class', 
    'Territory'
])['Observation'].sum().reset_index()

# Add an "All Regions" option by summing across regions
all_regions_data = grouped_data.groupby([
    'TIME_PERIOD', 
    'Result (DESC)', 
    'Person class'
])['Observation'].sum().reset_index()
all_regions_data['Territory'] = 'All Regions'

# Combine both datasets
final_data = pd.concat([grouped_data, all_regions_data], ignore_index=True)

print(f"Data prepared: {final_data.shape[0]} rows")
print(f"\nSample data:")
print(final_data.head(10))

Data prepared: 2051 rows

Sample data:
   TIME_PERIOD Result (DESC) Person class                           Territory  \
0         2010       Injured       Driver  'Valle d"'Aosta / Vallée d"'Aoste'   
1         2010       Injured       Driver                             Abruzzo   
2         2010       Injured       Driver                          Basilicata   
3         2010       Injured       Driver                            Calabria   
4         2010       Injured       Driver                            Campania   
5         2010       Injured       Driver                      Emilia-Romagna   
6         2010       Injured       Driver               Friuli-Venezia Giulia   
7         2010       Injured       Driver                               Lazio   
8         2010       Injured       Driver                             Liguria   
9         2010       Injured       Driver                           Lombardia   

   Observation  
0          346  
1         4338  
2         1192  
3

In [19]:
# Prepare population data - get total population per region per year
pop_filtered = population_df[
    (population_df['Gender'] == 'Total') &
    (population_df['Age (DESC)'] == 'Total') &
    (population_df['Marital status'] == 'Total')
][['Territory', 'TIME_PERIOD', 'Observation']].copy()

pop_filtered = pop_filtered.rename(columns={'Observation': 'Population'})

# BACKFILL POPULATION DATA FOR 2010-2018
# Since population data is only available for 2019-2025, we backfill 2010-2018 
# using the 2019 population values for each region (assuming relatively stable population)
pop_2019 = pop_filtered[pop_filtered['TIME_PERIOD'] == 2019].copy()

backfilled_data = []
for year in range(2010, 2019):
    year_data = pop_2019.copy()
    year_data['TIME_PERIOD'] = year
    backfilled_data.append(year_data)

backfilled_df = pd.concat(backfilled_data, ignore_index=True)

# Combine backfilled data with original population data
pop_filtered = pd.concat([backfilled_df, pop_filtered], ignore_index=True).sort_values(['Territory', 'TIME_PERIOD']).reset_index(drop=True)

# Calculate total population for all of Italy by year
italy_total_pop = pop_filtered.groupby('TIME_PERIOD')['Population'].sum().reset_index()
italy_total_pop['Territory'] = 'All Regions'

# Combine regional and total population
population_data = pd.concat([pop_filtered, italy_total_pop], ignore_index=True)

print(f"Population data prepared: {population_data.shape[0]} rows")
print(f"\nYear range: {population_data['TIME_PERIOD'].min()} - {population_data['TIME_PERIOD'].max()}")
print(f"Note: Years 2010-2018 use backfilled 2019 population values")
print("\nSample population data:")
print(population_data.head(10))
print("\nAll Regions total:")
print(population_data[population_data['Territory'] == 'All Regions'])

Population data prepared: 368 rows

Year range: 2010 - 2025
Note: Years 2010-2018 use backfilled 2019 population values

Sample population data:
                            Territory  TIME_PERIOD  Population
0  'Valle d"'Aosta / Vallée d"'Aoste'         2010      125653
1  'Valle d"'Aosta / Vallée d"'Aoste'         2011      125653
2  'Valle d"'Aosta / Vallée d"'Aoste'         2012      125653
3  'Valle d"'Aosta / Vallée d"'Aoste'         2013      125653
4  'Valle d"'Aosta / Vallée d"'Aoste'         2014      125653
5  'Valle d"'Aosta / Vallée d"'Aoste'         2015      125653
6  'Valle d"'Aosta / Vallée d"'Aoste'         2016      125653
7  'Valle d"'Aosta / Vallée d"'Aoste'         2017      125653
8  'Valle d"'Aosta / Vallée d"'Aoste'         2018      125653
9  'Valle d"'Aosta / Vallée d"'Aoste'         2019      125653

All Regions total:
       Territory  TIME_PERIOD  Population
352  All Regions         2010    60890707
353  All Regions         2011    60890707
354  All Regions

In [22]:
population_data

,Territory,TIME_PERIOD,Population
0,"'Valle d""'Aosta / Vallée d""'Aoste'",2010,125653
1,"'Valle d""'Aosta / Vallée d""'Aoste'",2011,125653
2,"'Valle d""'Aosta / Vallée d""'Aoste'",2012,125653
3,"'Valle d""'Aosta / Vallée d""'Aoste'",2013,125653
4,"'Valle d""'Aosta / Vallée d""'Aoste'",2014,125653
...,...,...,...
363,All Regions,2021,60313291
364,All Regions,2022,60103707
365,All Regions,2023,60074344
366,All Regions,2024,60053932


In [20]:
# Merge accident data with population data
enhanced_data = final_data.merge(
    population_data,
    on=['Territory', 'TIME_PERIOD'],
    how='left'
)

# Calculate per 100,000 inhabitants
enhanced_data['Per_100k'] = (enhanced_data['Observation'] / enhanced_data['Population']) * 100000

# Create Total Counts version - include ALL years (2010-2024)
count_data = enhanced_data.copy()
count_data['Metric_Type'] = 'Total Counts'
count_data['Value'] = count_data['Observation']

# Create Per 100,000 version - now includes all years (2010-2024) with backfilled population data
rate_data = enhanced_data[enhanced_data['Population'].notna()].copy()
rate_data['Metric_Type'] = 'Per 100,000 Inhabitants'
rate_data['Value'] = rate_data['Per_100k']

# Combine both versions
chart_data_enhanced = pd.concat([count_data, rate_data], ignore_index=True)

print(f"Enhanced data prepared: {chart_data_enhanced.shape[0]} rows")
print(f"\nTotal Counts - Years available: {sorted(chart_data_enhanced[chart_data_enhanced['Metric_Type'] == 'Total Counts']['TIME_PERIOD'].unique())}")
print(f"Per 100k - Years available: {sorted(chart_data_enhanced[chart_data_enhanced['Metric_Type'] == 'Per 100,000 Inhabitants']['TIME_PERIOD'].unique())}")
print("\nSample Total Counts data (All Regions, 2010-2012):")
print("Note: Per 100k rates for 2010-2018 use backfilled 2019 population values")
print("\nSample Total Counts data (All Regions, 2010-2012):")
print(chart_data_enhanced[
    (chart_data_enhanced['Territory'] == 'All Regions') & 
    (chart_data_enhanced['Metric_Type'] == 'Total Counts') &
    (chart_data_enhanced['TIME_PERIOD'].isin([2010, 2011, 2012]))
].head(10))
print("\nSample Per 100k data (All Regions, 2010-2012):")
print(chart_data_enhanced[
    (chart_data_enhanced['Territory'] == 'All Regions') & 
    (chart_data_enhanced['Metric_Type'] == 'Per 100,000 Inhabitants') &    (chart_data_enhanced['TIME_PERIOD'].isin([2010, 2011, 2012]))
].head(10))

Enhanced data prepared: 4102 rows

Total Counts - Years available: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Per 100k - Years available: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Sample Total Counts data (All Regions, 2010-2012):
Note: Per 100k rates for 2010-2018 use backfilled 2019 population values

Sample Total Counts data (All Regions, 2010-2012):
      TIME_PERIOD Result (DESC) Person class    Territory  Observation  \
1961         2010       Injured       Driver  All Regions       211885   
1962         2010       Injured    Passenger  All Regions        74513   
1963         2010       

In [21]:
# Create enhanced interactive Altair chart with all dropdown menus

# Create dropdown for region selection
region_dropdown = alt.binding_select(
    options=['All Regions'] + sorted(chart_data_enhanced[chart_data_enhanced['Territory'] != 'All Regions']['Territory'].unique().tolist()),
    name='Select Region: '
)
region_selection = alt.selection_point(
    fields=['Territory'],
    bind=region_dropdown,
    value='All Regions'
)

# Create dropdown for metric selection (Killed or Injured)
metric_dropdown = alt.binding_select(
    options=['Killed', 'Injured'],
    name='Select Metric: '
)
metric_selection = alt.selection_point(
    fields=['Result (DESC)'],
    bind=metric_dropdown,
    value='Injured'
)

# Create dropdown for display type (Total Counts or Per 100,000)
display_dropdown = alt.binding_select(
    options=['Total Counts', 'Per 100,000 Inhabitants'],
    name='Display As: '
)
display_selection = alt.selection_point(
    fields=['Metric_Type'],
    bind=display_dropdown,
    value='Total Counts'
)

# Create the stacked bar chart with dynamic y-axis
chart_enhanced = alt.Chart(chart_data_enhanced).mark_bar().encode(
    x=alt.X('TIME_PERIOD:O', 
            title='Year',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Value:Q', 
            title='Number of People',
            stack='zero'),
    color=alt.Color('Person class:N',
                   title='Person Type',
                   scale=alt.Scale(
                       domain=['Driver', 'Passenger', 'Pedestrian'],
                       range=['#4169E1', '#FF6B6B', '#2ECC71']
                   )),
    tooltip=[
        alt.Tooltip('TIME_PERIOD:O', title='Year'),
        alt.Tooltip('Person class:N', title='Person Type'),
        alt.Tooltip('Result (DESC):N', title='Metric'),
        alt.Tooltip('Territory:N', title='Region'),
        alt.Tooltip('Metric_Type:N', title='Display Type'),
        alt.Tooltip('Observation:Q', title='Count', format=','),
        alt.Tooltip('Value:Q', title='Value', format=',.2f')
    ]
).add_params(
    region_selection,
    metric_selection,
    display_selection
).transform_filter(
    region_selection
).transform_filter(
    metric_selection
).transform_filter(
    display_selection
).properties(
    width=700,
    height=400,
    title='Road Accident Casualties in Italy by Person Type and Year'
)

chart_enhanced

alt.Chart(...)

In [ ]:
# Download GeoJSON map of Italy regions
# URL to the Italy regions GeoJSON from openpolis/geojson-italy repository
geojson_url = 'https://raw.githubusercontent.com/openpolis/geojson-italy/master/geojson/limits_IT_regions.geojson'

# Define output path in the data folder
output_path = '../data/italy_regions.geojson'

# Download the GeoJSON file with User-Agent header to avoid connection issues
print(f"Downloading Italy regions GeoJSON from: {geojson_url}")
try:
    # Create a request with User-Agent header
    req = urllib.request.Request(
        geojson_url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    )
    
    # Download the file
    with urllib.request.urlopen(req) as response:
        geojson_content = response.read()
    
    # Save to file
    with open(output_path, 'wb') as f:
        f.write(geojson_content)
    
    print(f"Successfully downloaded to: {output_path}")
    
    # Load and display basic info about the GeoJSON
    with open(output_path, 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    
    print(f"\nGeoJSON contains {len(geojson_data['features'])} regions")
    print("\nRegion names in GeoJSON:")
    for feature in geojson_data['features']:
        print(f"  - {feature['properties'].get('reg_name', 'N/A')}")
        
except Exception as e:
    print(f"Error downloading file: {e}")

Error downloading file: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
